In [39]:
import pandas as pd
import numpy as np
import joblib

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

from mordred import Calculator, descriptors

In [40]:
pipeline = joblib.load("best_lgbm_pipeline.pkl")

final_feature_columns = joblib.load(
    "final_feature_columns.pkl"
)


In [ ]:
print(final_feature_columns)
print(len(final_feature_columns))

['DESC_ABC', 'DESC_ABCGG', 'DESC_nAcid', 'DESC_nBase', 'DESC_SpAbs_A', 'DESC_SpMax_A', 'DESC_SpDiam_A', 'DESC_SpAD_A', 'DESC_SpMAD_A', 'DESC_LogEE_A', 'DESC_VE1_A', 'DESC_VE2_A', 'DESC_VE3_A', 'DESC_VR1_A', 'DESC_VR2_A', 'DESC_VR3_A', 'DESC_nAromAtom', 'DESC_nAromBond', 'DESC_nAtom', 'DESC_nHeavyAtom', 'DESC_nSpiro', 'DESC_nBridgehead', 'DESC_nHetero', 'DESC_nH', 'DESC_nB', 'DESC_nC', 'DESC_nN', 'DESC_nO', 'DESC_nS', 'DESC_nP', 'DESC_nF', 'DESC_nCl', 'DESC_nBr', 'DESC_nI', 'DESC_nX', 'DESC_ATS0dv', 'DESC_ATS1dv', 'DESC_ATS2dv', 'DESC_ATS3dv', 'DESC_ATS4dv', 'DESC_ATS5dv', 'DESC_ATS6dv', 'DESC_ATS7dv', 'DESC_ATS8dv', 'DESC_ATS0d', 'DESC_ATS1d', 'DESC_ATS2d', 'DESC_ATS3d', 'DESC_ATS4d', 'DESC_ATS5d', 'DESC_ATS6d', 'DESC_ATS7d', 'DESC_ATS8d', 'DESC_ATS0s', 'DESC_ATS1s', 'DESC_ATS2s', 'DESC_ATS3s', 'DESC_ATS4s', 'DESC_ATS5s', 'DESC_ATS6s', 'DESC_ATS7s', 'DESC_ATS8s', 'DESC_ATS0Z', 'DESC_ATS1Z', 'DESC_ATS2Z', 'DESC_ATS3Z', 'DESC_ATS4Z', 'DESC_ATS5Z', 'DESC_ATS6Z', 'DESC_ATS7Z', 'DESC_ATS8Z'

: 

In [42]:
remover = SaltRemover.SaltRemover()

largest_fragment_chooser = (
    rdMolStandardize.LargestFragmentChooser()
)

def clean_molecule(smiles):

    try:

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        mol = remover.StripMol(mol)

        mol = largest_fragment_chooser.choose(
            mol
        )

        Chem.SanitizeMol(mol)

        return mol

    except:
        return None

In [43]:
calc = Calculator(
    descriptors,
    ignore_3D=True
)

def generate_descriptors(mol):

    desc_df = calc.pandas([mol])

    desc_df = desc_df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    return desc_df

In [44]:
def generate_fingerprint(mol):

    fp = list(

        AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=1024
        )

    )

    fp_df = pd.DataFrame([fp])

    fp_df.columns = [

        f"FPMorgan_{i}"

        for i in range(1024)

    ]

    return fp_df

In [45]:
def predict_solubility(smiles):

    mol = clean_molecule(smiles)

    if mol is None:

        return "Invalid SMILES"

    # descriptors

    descriptor_df = generate_descriptors(mol)

    descriptor_df.columns = [

        f"DESC_{col}"

        for col in descriptor_df.columns

    ]

    # fingerprints

    fingerprint_df = generate_fingerprint(mol)

    # combine

    X_new = pd.concat(

        [descriptor_df, fingerprint_df],

        axis=1

    )

    # align columns

    X_new = X_new.reindex(

        columns=final_feature_columns,

        fill_value=np.nan

    )
    


    # prediction

    pred = pipeline.predict(X_new)

    return pred[0]

In [46]:
smiles = "O=Cc1ccc(Cl)cc1"

prediction = predict_solubility(
    smiles
)

print(
    f"Predicted Solubility: {prediction}"
)

[15:19:27] Running LargestFragmentChooser
100%|██████████| 1/1 [00:01<00:00,  1.16s/it]


Predicted Solubility: -2.1539698571044483


[15:19:31] DEPRECATION WARNING: please use MorganGenerator
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [47]:
smiles_list = [

    "CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-]",
    "O=C1Nc2cccc3ccccc123",
    "O=Cc1ccc(Cl)cc1"

]

for smi in smiles_list:

    pred = predict_solubility(smi)

    print(
        f"{smi} --> {pred}"
    )

[15:19:36] Running LargestFragmentChooser
100%|██████████| 1/1 [00:01<00:00,  1.16s/it]
[15:19:39] DEPRECATION WARNING: please use MorganGenerator
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[15:19:39] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 11
[15:19:39] Running LargestFragmentChooser


CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-] --> -3.5861944575689813
O=C1Nc2cccc3ccccc123 --> Invalid SMILES


100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


O=Cc1ccc(Cl)cc1 --> -2.1539698571044483


[15:19:42] DEPRECATION WARNING: please use MorganGenerator
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
